# Emotion Detection

Emotion detection is a field of study that aims to identify and classify human emotions from various forms of data, such as text, speech, facial expressions, and physiological signals. It has applications in areas like human-computer interaction, sentiment analysis, mental health monitoring, and personalized content recommendation.

In natural language processing (NLP), emotion detection from text involves analyzing the words, phrases, and sentence structures to infer the emotional state of the writer. This can be a challenging task due to the subjective nature of emotions, the nuances of language, and the possibility of sarcasm or irony.

Common approaches to emotion detection from text include:

*   **Lexicon-based methods:** These methods use predefined lists of words associated with different emotions and calculate an overall emotion score based on the presence and intensity of these words in the text.
*   **Machine learning methods:** These methods train models on labeled datasets of text and corresponding emotions. Various algorithms, such as Naive Bayes, Support Vector Machines (SVMs), and deep learning models (like Recurrent Neural Networks (RNNs) and Transformers), can be used for this task.
*   **Hybrid approaches:** These methods combine lexicon-based and machine learning techniques to leverage the strengths of both approaches.

The performance of emotion detection models depends heavily on the quality and size of the training data, the complexity of the emotions being classified, and the chosen features and algorithms.

# Emotion Detection using Bidirectional LSTM

Bidirectional Long Short-Term Memory (BiLSTM) networks are a type of recurrent neural network (RNN) that are particularly effective for sequence processing tasks like emotion detection from text. Unlike traditional LSTMs that process sequences in one direction (either forward or backward), BiLSTMs process the sequence in both directions simultaneously.

Here's how BiLSTMs are used for emotion detection:

1.  **Forward Pass:** An LSTM layer processes the input text sequence from the beginning to the end.
2.  **Backward Pass:** Another LSTM layer processes the input text sequence from the end to the beginning.
3.  **Concatenation:** The outputs from the forward and backward LSTM layers are concatenated at each time step. This allows the model to have a comprehensive understanding of the context from both past and future words in the sequence.

By considering both past and future context, BiLSTMs can capture more nuanced information and dependencies within the text, which is crucial for accurately identifying emotions. For example, the meaning of a word and its emotional connotation can be influenced by the words that come after it.

In the context of emotion detection, the output of the BiLSTM layer is typically fed into a dense layer with a softmax activation function. The dense layer learns to map the rich representations generated by the BiLSTM to the different emotion categories, and the softmax function outputs the probability distribution over these categories.

This approach allows the model to leverage the sequential nature of text data and the power of LSTMs to capture long-range dependencies, leading to improved performance in emotion detection tasks.



[Emotion Dection using Bidirectional ](https://www.geeksforgeeks.org/machine-learning/emotion-detection-using-bidirectional-lstm/)

## Step 1: Importing the required libraries

In [ ]:
import keras
import numpy as np
from keras.models import Sequential
from keras.layers import Dense, Bidirectional, LSTM, Embedding, SpatialDropout1D
# Correcting the import paths for Tokenizer and pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from nltk.tokenize import word_tokenize
import pandas as pd
import nltk
from sklearn.model_selection import train_test_split
import requests

# Download necessary NLTK resources
nltk.download('punkt')
nltk.download('punkt_tab')

## Step 2: Load the dataset

In [ ]:

# Download the file
file_id = '1Y0fK6M3XWvClcB5JdPYLBZkDq0ErmMPe'
url = f'https://drive.google.com/uc?id={file_id}&export=download'

response = requests.get(url, stream=True)
with open('isear.csv', 'wb') as f:
    for chunk in response.iter_content(chunk_size=8192):
        if chunk:
            f.write(chunk)

try:
    df = pd.read_csv('isear.csv', header=None)
    display(df.head())
    df.drop(df[df[1] == '[ No response.]'].index, inplace=True)
except FileNotFoundError:
    print("Error: 'isear.csv' not found after download attempt.")


## Step 3: Tokenization


In [ ]:
feel_arr = df[1]
feel_arr = [word_tokenize(sent) for sent in feel_arr]

## Step 4: Padding


In [ ]:
max_words = 100
max_len = 100
tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(feel_arr)
X = tokenizer.texts_to_sequences(feel_arr)
X = pad_sequences(X, padding='post', maxlen=max_len)

## Step 5: Prepare labels


In [ ]:
labels = df[0].values
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y = le.fit_transform(labels)

## Step 6: Train-test split


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## Step 7: Build the BiLSTM model


In [ ]:
model = Sequential()
model.add(Embedding(input_dim=max_words, output_dim=128, input_length=max_len))
model.add(SpatialDropout1D(0.2))
model.add(Bidirectional(LSTM(100, dropout=0.2, recurrent_dropout=0.2)))
model.add(Dense(7, activation='softmax'))  # 7 emotions

## Step 8: Compile the model


In [ ]:
model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

## Step 9: Train the model


In [ ]:
history = model.fit(X_train, y_train, epochs=5, batch_size=64, validation_data=(X_test, y_test), verbose=2)

## Step 10: Evaluate the model


In [ ]:
loss, accuracy = model.evaluate(X_test, y_test)
print(f"Test Accuracy: {accuracy * 100:.2f}%")

In [2]:
# !pip install transformers datasets scikit-learn tensorflow torch --upgrade

In [3]:
# import tensorflow as tf
# from transformers import AutoTokenizer, TFAutoModelForSequenceClassification
# from sklearn.preprocessing import LabelEncoder
# from sklearn.model_selection import train_test_split
# from tensorflow.keras.optimizers import Adam
# import pandas as pd
# import requests

# # Download the dataset from Google Drive
# file_id = '1Y0fK6M3XWvClcB5JdPYLBZkDq0ErmMPe'
# url = f'https://drive.google.com/uc?id={file_id}&export=download'

# response = requests.get(url, stream=True)
# with open('isear.csv', 'wb') as f:
#     for chunk in response.iter_content(chunk_size=8192):
#         if chunk:
#             f.write(chunk)

# # Load the dataset
# df = pd.read_csv('isear.csv', header=None)
# df.drop(df[df[1] == '[ No response.]'].index, inplace=True)

# # Encode labels
# le = LabelEncoder()
# df[0] = le.fit_transform(df[0])
# num_classes = len(le.classes_)

# # Train-test split
# train_texts, test_texts, train_labels, test_labels = train_test_split(df[1].values, df[0].values, test_size=0.2, random_state=42)

# # Load tokenizer and model
# tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
# model = TFAutoModelForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=num_classes)

# # Tokenize
# train_encodings = tokenizer(list(train_texts), truncation=True, padding=True, max_length=128)
# test_encodings = tokenizer(list(test_texts), truncation=True, padding=True, max_length=128)

# # Convert to TensorFlow datasets
# train_dataset = tf.data.Dataset.from_tensor_slices((
#     dict(train_encodings),
#     train_labels
# )).shuffle(1000).batch(16)

# test_dataset = tf.data.Dataset.from_tensor_slices((
#     dict(test_encodings),
#     test_labels
# )).batch(16)

# # Compile the model
# optimizer = Adam(learning_rate=5e-5)
# loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
# metric = tf.keras.metrics.SparseCategoricalAccuracy('accuracy')

# model.compile(optimizer=optimizer, loss=loss, metrics=[metric])

# # Train the model
# history = model.fit(train_dataset, validation_data=test_dataset, epochs=3)

# # Evaluate the model
# loss_val, accuracy_val = model.evaluate(test_dataset)
# print(f"Transformer-based Model Accuracy: {accuracy_val * 100:.2f}%")


# Emotion Detection using `bert-base-uncased`

Using a pre-trained transformer model like BERT (`bert-base-uncased`) for emotion detection leverages the powerful language understanding capabilities learned during its extensive pre-training on a massive text corpus. Instead of training a model from scratch, we fine-tune BERT on our specific emotion detection dataset.

Here's a general outline of the process when using `bert-base-uncased`:

1.  **Load the pre-trained tokenizer:** The `bert-base-uncased` model comes with its own tokenizer that needs to be used to process the input text. This tokenizer handles tasks like converting words into numerical IDs, adding special tokens (like `[CLS]` for classification and `[SEP]` to separate sentences), and creating attention masks.
2.  **Tokenize the input data:** The text data is tokenized using the loaded BERT tokenizer. This involves converting the text into a format that the BERT model can understand, including handling padding and truncation to ensure all input sequences have the same length.
3.  **Load the pre-trained BERT model for sequence classification:** We load the `bert-base-uncased` model with a classification head on top. This pre-trained model already has a deep understanding of language, and the classification head is a new layer that will be trained to predict the emotion categories.
4.  **Fine-tune the model:** The loaded model is then fine-tuned on the labeled emotion detection dataset. This involves training the model for a few epochs with a relatively small learning rate. During fine-tuning, the model adjusts its parameters (both the pre-trained BERT layers and the new classification head) to better perform the emotion detection task.
5.  **Prediction:** Once fine-tuned, the model can be used to predict the emotion of new, unseen text data. The output of the classification head is typically a probability distribution over the emotion categories.

The advantage of using a pre-trained model like BERT is that it significantly reduces the amount of data and computational resources required for training compared to building a model from scratch. BERT's ability to capture contextual relationships between words makes it particularly effective for tasks like emotion detection where the meaning and emotional tone of words can depend heavily on their surrounding context.

In [4]:
!pip install torch torchvision torchaudio transformers --upgrade


In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.optim import Adam
from torch.utils.data import DataLoader, Dataset
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Load dataset (example with ISEAR dataset)
file_id = '1Y0fK6M3XWvClcB5JdPYLBZkDq0ErmMPe'
url = f'https://drive.google.com/uc?id={file_id}&export=download'
df = pd.read_csv(url, header=None)
df.columns = ['Emotion', 'Sentence']
df = df[df['Sentence'] != '[ No response.]']

# Encode labels
le = LabelEncoder()
df['Emotion'] = le.fit_transform(df['Emotion'])
num_classes = len(le.classes_)

# Split data
train_texts, test_texts, train_labels, test_labels = train_test_split(df['Sentence'].tolist(), df['Emotion'].tolist(), test_size=0.2, random_state=42)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

# Tokenize the datasets
train_encodings = tokenizer(train_texts, truncation=True, padding=True, max_length=128)
test_encodings = tokenizer(test_texts, truncation=True, padding=True, max_length=128)

# Create Dataset class
class EmotionDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item

train_dataset = EmotionDataset(train_encodings, train_labels)
test_dataset = EmotionDataset(test_encodings, test_labels)

# Load PyTorch model
model = AutoModelForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=num_classes)

# Define optimizer and loss function
optimizer = Adam(model.parameters(), lr=5e-5)
criterion = torch.nn.CrossEntropyLoss()

# Prepare DataLoader
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)

# Move model to device
device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
model.to(device)

# Training loop
model.train()
for epoch in range(1):
    total_loss = 0
    for batch in train_loader:
        optimizer.zero_grad()
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    print(f'Epoch {epoch+1}, Loss: {total_loss / len(train_loader)}')

# Evaluation
model.eval()
all_preds = []
all_labels = []
with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        preds = torch.argmax(outputs.logits, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

acc = accuracy_score(all_labels, all_preds)
print(f"Test Accuracy: {acc * 100:.2f}%")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Epoch 1, Loss: 1.1447297314226155
Test Accuracy: 68.51%
